# 🧠💜 BrainFormer v13-v3 — C/CUDA Deep Brain + Analytics (Kaggle T4×2)

Pure **C/CUDA** brain (custom kernels, nvcc sm_75). Python only tokenizes data and runs the LLM opponent. Goal: **maximum tokens/sec** — and deep analytics to see bottlenecks + quality.

**Status:** M1–M5 ✓ — depth + learned mixture, competitive granules, atomic-free bucketed scatter (GPU prefix-sum), resident-VRAM shard, **cuBLAS tensor activation**, **device-side mixture** (no per-step sync), **lag-free analytics** (profiled after training), and **dual-GPU local-SGD**.

Pipeline: clone → build data → LLM ppl → `nvcc` → smoke → **dual-GPU train** → analytics → knob sweep.

**Settings:** Accelerator `GPU T4 x2`, Internet `On`, add Kaggle Secret `GITHUB_TOKEN`.

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient
# absolute paths + chdir to a fixed root so RE-RUNNING this cell never nests
ROOT = '/kaggle/working'
REPO = ROOT + '/plango-labs'
DIR = REPO + '/test_013_v3'
os.chdir(ROOT)
tok = UserSecretsClient().get_secret('GITHUB_TOKEN')
url = f'https://{tok}@github.com/PlangoDev/plango-labs.git'
try:
    if not os.path.isdir(REPO):
        subprocess.run(['git', 'clone', '--depth', '1', url, REPO], check=True, capture_output=True)
    else:
        subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=True, capture_output=True)
except subprocess.CalledProcessError:
    raise RuntimeError('git clone/pull failed - check the GITHUB_TOKEN secret') from None
finally:
    del tok, url
os.chdir(DIR)
print('cwd:', os.getcwd())
!nvcc --version | tail -2 && nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Build data (Python, Internet ON) — train + WikiText-103 test bins + meta.json

In [ ]:
!python tools/build_data.py --model EleutherAI/pythia-410m --train-tokens 50000000 --out-dir /kaggle/working

## 2. LLM opponent perplexity (Python, GPU) — the only torch in v3

In [ ]:
!python tools/run_llm.py --meta /kaggle/working/meta.json --out /kaggle/working/llm_result.json

## 3. Compile (nvcc, sm_75)

In [ ]:
!make clean && make 2>&1 | tail -25 && echo '--- built ---' && ls -la brain

## 4. Smoke test (tiny config, atomic scatter) — finite scoreboard = pipeline OK

In [ ]:
!./brain --smoke --meta /kaggle/working/meta.json --out /kaggle/working/brain_smoke.json

## 5. Train + eval (speed-first: 2 layers, n_classes=2048, bucketed scatter, resident shard, DUAL-GPU)

`--n-gpus 2` uses both T4s (data-parallel local-SGD). Watch the live purple progress bar + tok/s, then the per-phase timing + granule health at the end.
Speed dials: `--n-classes ↑`, `--n-layers ↓`, `--k-active ↓`. Quality config: `--n-layers 4 --n-classes 256`.
Fallbacks if anything misbehaves: `--n-gpus 1`, `--fast-scatter 0`, `--cublas 0`, `--resident 0`.

In [ ]:
!./brain --meta /kaggle/working/meta.json --out /kaggle/working/brain_result.json --train-tokens 50000000 --n-gpus 2

## 6. Scoreboard

In [ ]:
!python tools/scoreboard.py --brain /kaggle/working/brain_result.json --llm /kaggle/working/llm_result.json

## 7. Analytics — where the time goes (bottlenecks) and code health (quality)

Reads `brain_result.json`. The per-phase bars show which kernel dominates wall-clock; granule health (dead %, usage-Gini, mean margin) shows whether the sparse code is healthy or wasted.

In [ ]:
import json
import matplotlib.pyplot as plt
b = json.load(open('/kaggle/working/brain_result.json'))
l = json.load(open('/kaggle/working/llm_result.json'))

print(f"throughput     : {b['tok_per_sec']:,.0f} tok/s  ({b['consumed_tokens']:,} tok / {b['train_seconds']:.0f}s)")
print(f"brain ppl      : {b['brain_ppl']:.2f}   vs LLM {l['llm_ppl']:.2f}")
print(f"MACs/token     : {b['macs_per_token']:,}  ({b['macs_per_token']/l['llm_macs']:.5f}x LLM)")
print(f"scatter        : {b.get('scatter','?')}   resident: {b.get('resident','?')}")

ph = b.get('phase_pct', {})
if ph:
    items = sorted(ph.items(), key=lambda kv: kv[1])
    names = [k for k, _ in items]; vals = [v for _, v in items]
    plt.figure(figsize=(8, 3.5))
    plt.barh(names, vals, color='#4C9AFF')
    plt.xlabel('% of measured training time'); plt.title('Per-phase timing (bottlenecks)')
    for i, v in enumerate(vals):
        plt.text(v + 0.5, i, f'{v:.1f}%', va='center')
    plt.tight_layout(); plt.show()
    top = items[-1]
    print(f"\nbottleneck: {top[0]} ({top[1]:.1f}% of measured time)")

h = b.get('health', {})
if h:
    print(f"\ngranule health (layer 0):")
    print(f"  dead granules : {100*h['dead_frac']:.1f}%  (lower is better; high = wasted capacity)")
    print(f"  usage Gini    : {h['usage_gini']:.3f}  (0=even load, 1=a few hog the code)")
    print(f"  mean margin   : {h['mean_margin']:.3f}  (kWTA code strength)")

## 8. Per-layer quality + learned mixture

Does depth help? If deep layers earn mixture weight and have lower ppl, the stack is working. If the mixture dumps everything on layer 0, the relay isn't carrying signal (→ try a learned relay next).

In [ ]:
import json
import matplotlib.pyplot as plt
b = json.load(open('/kaggle/working/brain_result.json'))
pl = b.get('per_layer_ppl', []); mx = b.get('mix', [])
for i, (p, w) in enumerate(zip(pl, mx)):
    print(f"  layer {i}: ppl {p:>10.2f}   mixture weight {w:.3f}")
if pl:
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
    ax[0].bar(range(len(pl)), pl, color='#36B37E'); ax[0].set_title('per-layer perplexity'); ax[0].set_xlabel('layer')
    ax[1].bar(range(len(mx)), mx, color='#FF8B00'); ax[1].set_title('learned mixture weight'); ax[1].set_xlabel('layer')
    plt.tight_layout(); plt.show()

## 9. Throughput knob sweep (`--bench`, no eval) — find the fastest config

Each runs ~a few seconds of training and reports tok/s. Sweeps the dials that set readout traffic: `n_classes` (↑→small S→fast), `n_layers`, `k_active`, and atomic vs bucketed scatter.

In [ ]:
import subprocess, json, re
configs = [
    ('4L  n_classes=256  (quality)', ['--n-layers','4','--n-classes','256']),
    ('2L  n_classes=2048 (default)', ['--n-layers','2','--n-classes','2048']),
    ('2L  n_classes=4096',          ['--n-layers','2','--n-classes','4096']),
    ('1L  n_classes=4096',          ['--n-layers','1','--n-classes','4096']),
    ('1L  n_classes=4096 K=32',     ['--n-layers','1','--n-classes','4096','--k-active','32']),
    ('2L  n_classes=2048 atomic',   ['--n-layers','2','--n-classes','2048','--fast-scatter','0']),
]
rows = []
for name, extra in configs:
    cmd = ['./brain','--bench','--meta','/kaggle/working/meta.json',
           '--train-tokens','3000000','--analytics','0'] + extra
    out = subprocess.run(cmd, capture_output=True, text=True).stdout
    m = re.findall(r'([0-9][0-9,]*)\s*tok/s', out)
    tps = m[-1].replace(',','') if m else '?'
    rows.append((name, tps))
    print(f"  {name:<32} {tps:>12} tok/s")
print('\n(quality vs speed: compare these tok/s against the perplexity each config reaches in a full run.)')

## 10. Quality experiments — close the perplexity gap

The speed config (n_classes=2048) trades quality for speed, and the random relay means depth isn't helping yet (layers come out tied). These two runs test the fixes: **(1)** the quality config (n_classes=256, 4 layers), and **(2)** `--boost 1`, which makes each layer specialize on what the layer below got wrong (residual boosting, no backprop). Watch whether boosting makes deeper layers earn lower ppl + more mixture weight.

In [ ]:
# ── QUALITY experiments (close the perplexity gap) ──────────────────────────
# 1) Quality config: low n_classes recovers the hierarchical tax; 4 layers for depth.
!./brain --meta /kaggle/working/meta.json --out /kaggle/working/brain_quality.json \
    --train-tokens 50000000 --n-gpus 2 --n-layers 4 --n-classes 256

# 2) Boosting A/B — does it make depth ADDITIVE? (layer l learns what l-1 missed)
#    Compare per-layer ppl + mixture weights vs run (1): if boosting works, deeper
#    layers get LOWER ppl and EARN more mixture weight.
!./brain --meta /kaggle/working/meta.json --out /kaggle/working/brain_boost.json \
    --train-tokens 50000000 --n-gpus 2 --n-layers 4 --n-classes 256 --boost 1

import json
for tag, path in [('no-boost', '/kaggle/working/brain_quality.json'),
                  ('boost',    '/kaggle/working/brain_boost.json')]:
    b = json.load(open(path))
    pl = ' '.join(f'{p:.0f}' for p in b['per_layer_ppl'])
    mx = ' '.join(f'{w:.2f}' for w in b['mix'])
    print(f"{tag:>9}: ppl {b['brain_ppl']:.1f}  | per-layer [{pl}]  mix [{mx}]  {b['tok_per_sec']:,.0f} tok/s")

## 11. Test-time adaptation (hyper-fixation) + generation — the table-turner

`--adapt` lets the brain keep learning *as it reads the test doc* (predict-then-learn, no leakage). The frozen LLM can't. Watch static→adaptive ppl drop. `--gen-tokens` then samples a continuation so you can *see* it (detok in Python). At high ppl expect locally-plausible word flow, not coherence yet — the baseline to watch improve.

In [ ]:
# adaptive eval (--adapt) + generate 120 tokens (--gen-tokens), then detokenize
!./brain --meta /kaggle/working/meta.json --out /kaggle/working/brain_adapt.json \
    --n-gpus 2 --train-tokens 50000000 --n-classes 256 \
    --adapt 1 --adapt-lr 0.1 --gen-tokens 120 --gen-temp 0.8 \
    --gen-out /kaggle/working/brain_gen.json
print()
!python tools/detok.py --meta /kaggle/working/meta.json --gen /kaggle/working/brain_gen.json

## 10. Full 500M-token run (headline) — rebuild bins at full budget first

In [ ]:
# !python tools/build_data.py --train-tokens 500000000 --out-dir /kaggle/working
# !python tools/run_llm.py --meta /kaggle/working/meta.json --out /kaggle/working/llm_result.json
# !./brain --meta /kaggle/working/meta.json --out /kaggle/working/brain_result.json --train-tokens 500000000
# !python tools/scoreboard.py --brain /kaggle/working/brain_result.json --llm /kaggle/working/llm_result.json